# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**Franco Marcelo Morero**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [1]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi


In [2]:
# Heurística propia: Suma ponderada con detección de discos inestables.
#
# La idea clave es que mover un disco grande es exponencialmente más costoso
# que mover uno pequeño. Para mover el disco i a su posición final, se necesitan
# al menos 2^(i-1) movimientos (hay que despejar los i-1 discos más pequeños).
#
# Un disco se considera "mal ubicado" si:
#   1. No está en la varilla objetivo (varilla 3), o
#   2. Está en la varilla objetivo pero es "inestable": algún disco más grande
#      que debería estar debajo de él no está en la varilla objetivo.
#      Esto significa que ese disco va a tener que moverse eventualmente.
#
# h(n) = Σ 2^(i-1) para cada disco i mal ubicado
#
# Esta heurística es admisible porque 2^(i-1) es una cota inferior del número
# de movimientos necesarios para colocar el disco i en su posición final.
# Para el estado inicial de 5 discos: h = 2^0+2^1+2^2+2^3+2^4 = 31,
# que coincide exactamente con el costo óptimo (2^n - 1).
# Esto se demuestra por inducción:
#   Caso base: 1 disco → 1 movimiento = 2^1 - 1.
#   Paso inductivo: para mover n discos de A a C se necesitan:
#     - 2^(n-1) - 1 movimientos para mover los n-1 discos de A a B,
#     - 1 movimiento para mover el disco n de A a C,
#     - 2^(n-1) - 1 movimientos para mover los n-1 discos de B a C.
#     Total: 2·(2^(n-1) - 1) + 1 = 2^n - 1.
#
# Ejemplos (5 discos, varilla objetivo = varilla 3):
#
#   Estado inicial: 5 4 3 2 1 | |
#     Varilla objetivo vacía, ningún disco estable.
#     h = 2^0+2^1+2^2+2^3+2^4 = 31 (= costo óptimo real)
#
#   Solo disco grande bien puesto: 4 3 2 1 | | 5
#     stable = {5}, mal ubicados = {1,2,3,4}
#     h = 1+2+4+8 = 15 (= costo de mover sub-torre de 4 discos: 2^4-1)
#
#   Disco pequeño sin base: 5 4 3 2 | | 1
#     Varilla objetivo = [1], expected empieza con [5,...] → 1 ≠ 5, break.
#     stable = {}, el disco 1 es INESTABLE (le faltan 5,4,3,2 debajo).
#     h = 1+2+4+8+16 = 31
#     La heurística CEIA contaría el disco 1 como "correcto" y daría h=4,
#     pero nuestra heurística detecta que es inestable y da h=31.
#
#   Base correcta, disco intruso arriba: 4 2 1 | | 5 3
#     Disco 5=5 → estable. Disco 3≠4 → break.
#     stable = {5}, mal ubicados = {1,2,3,4}
#     h = 1+2+4+8 = 15
#     El disco 3 está en la varilla objetivo pero es inestable.
#
def heuristic_weighted(state, goal_rod_index=2, number_disks=5):
    goal_rod = state.rods[goal_rod_index]

    # Determinamos qué discos están "establemente" colocados en la varilla objetivo.
    # Un disco es estable si todos los discos más grandes que él ya están
    # correctamente apilados debajo en la varilla objetivo.
    stable_on_goal = set()
    expected = list(range(number_disks, 0, -1))  # [5, 4, 3, 2, 1]
    for i, disk in enumerate(goal_rod):
        if disk == expected[i]:
            stable_on_goal.add(disk)
        else:
            # Si un disco no coincide, los de arriba tampoco son estables
            break

    # Sumamos 2^(i-1) por cada disco mal ubicado (no estable en la varilla objetivo)
    h = 0
    for disk in range(1, number_disks + 1):
        if disk not in stable_on_goal:
            h += 2 ** (disk - 1)

    return h

In [3]:
from aima_libs.aima import PriorityQueue as AimaPriorityQueue
from typing import Tuple, Callable

def search_algorithm(number_disks=5) -> Tuple[NodeHanoi | None, dict]:
    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    def f(node):
        return node.path_cost + heuristic_weighted(node.state, number_disks=number_disks)

    frontier = AimaPriorityQueue(order='min', f=f)
    frontier.append(NodeHanoi(initial_state))
    explored = set()
    nodes_explored = 0
    max_depth = 0

    # Inicializamos las salidas, pero reemplazar con lo que se quiera usar.
    metrics = {
        "solution_found": True,
        "nodes_explored": None,
        "states_visited": None,
        "nodes_in_frontier": None,
        "max_depth": None,
        "cost_total": None,
    }

    while len(frontier) != 0:
        _, node = frontier.pop()
        nodes_explored += 1
        max_depth = max(max_depth, node.depth)

        explored.add(node.state) # Agregamos el estado al set de explorados
        
        if problem.goal_test(node.state):
            metrics = {
                "solution_found": True,
                "nodes_explored": nodes_explored,
                "states_visited": len(explored),
                "nodes_in_frontier": len(frontier),
                "max_depth": node.depth,
                "cost_total": node.state.accumulated_cost,
            }
            return node, metrics
        
        # Agregamos a la frontera los nodos sucesores que no hayan sido visitados
        for child in node.expand(problem):
            if child.state not in explored:
                frontier.append(child)

    # Si no se encuentra solución, devolvemos métricas igualmente
    metrics = {
        "solution_found": False,
        "nodes_explored": nodes_explored,
        "states_visited": len(explored),
        "nodes_in_frontier": len(frontier),
        "max_depth": None,
        "cost_total": None,
    }
    return None, metrics

Se prueba la función:

In [4]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [5]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 170
states_visited: 119
nodes_in_frontier: 44
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [6]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [7]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
